# Root-Cause Ablation Suite

Notebook nay hien thuc truc tiep roadmap trong `docs/research/root_cause/teacher_root_cause_spec_2026-03-28.md`.

Muc tieu:

- khoa lai cac self-check de tranh sai mapping du lieu, remap target, oracle subset, va sampler
- evaluate baseline bang cung metric scale-aware + oracle-aligned
- chay cac ablation khuyen nghi:
  - `A1_curvature_compensated`
  - `A2_band_balanced`
  - `C1_scale800`
  - `C2_scale1200`
- xuat comparison table de so sanh truoc khi quyet dinh run tiep theo

Quy uoc quan trong:

- moi phep so sanh giua cac run co `target_scale` khac nhau deu duoc quy ve cung mien chuan `y600`
- metric trong "variant space" van duoc luu rieng de debug, nhung khong dung de chon winner

In [1]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\chess_engine")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "root_cause_ablation_suite"
RUN_DIR = Path(r"C:\Users\USER\Downloads\dgrn_5m_v3_stage2_polish_run1")
DATA_ROOT = PROJECT_ROOT / "data" / "process"

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import root_cause_ablation_helpers as lab

torch.set_float32_matmul_precision("high")
lab.set_global_seed(123)
paths = lab.build_default_paths(run_dir=RUN_DIR, data_root=DATA_ROOT, experiment_dir=EXPERIMENT_DIR)
lab.export_paths_json(paths, paths["output_dir"] / "paths.json")

if "envs\\chess_engine" not in sys.executable.lower():
    raise RuntimeError(
        f"Notebook is running under the wrong interpreter: {sys.executable}. "
        "Select the 'chess_engine' Jupyter kernel."
    )

DEVICE = lab.choose_device(prefer_cuda=True)
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is required for this notebook.")

CHECKPOINT = paths["run_dir"] / "ckpt_best.pt"
assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"

ORACLE_CFG = lab.OracleEvalConfig()
TRAIN_CFG = lab.TrainConfig(
    batch_size=640,
    epochs=1,
    learning_rate=3e-6,
    min_lr=1e-6,
    weight_decay=2e-4,
    grad_clip_norm=1.0,
    seed=123,
    log_every_steps=200,
    train_num_shards=None,
    val_max_samples=100_000,
    test_max_samples=200_000,
    val_num_shards=2,
    test_num_shards=4,
    benchmark_num_shards=1,
)

VARIANT_DICT = {variant.name: variant for variant in lab.build_recommended_variants()}
SELECTED_VARIANTS = [
    "A1_curvature_compensated",
    "A2_band_balanced",
    "C1_scale800",
    "C2_scale1200",
]

NOTEBOOK_CONFIG = {
    "oracle_cfg": asdict(ORACLE_CFG),
    "train_cfg": asdict(TRAIN_CFG),
    "selected_variants": SELECTED_VARIANTS,
}
lab.save_json(NOTEBOOK_CONFIG, paths["output_dir"] / "runtime_config.json")

train_cfg_validation = lab.validate_train_config(TRAIN_CFG)
oracle_cfg_validation = lab.validate_oracle_eval_config(ORACLE_CFG)
variant_validations = [lab.validate_variant_config(VARIANT_DICT[name]) for name in SELECTED_VARIANTS]
lab.save_json(train_cfg_validation, paths["reports_dir"] / "train_cfg_validation.json")
lab.save_json(oracle_cfg_validation, paths["reports_dir"] / "oracle_cfg_validation.json")
lab.save_json({"variants": variant_validations}, paths["reports_dir"] / "variant_cfg_validation.json")

print("python:", sys.executable)
print("device:", DEVICE)
print("gpu_name:", torch.cuda.get_device_name(0))
display(pd.DataFrame({"path_key": list(paths.keys()), "path_value": [str(v) for v in paths.values()]}))
display(pd.DataFrame(variant_validations))

python: c:\Users\USER\anaconda3\envs\chess_engine\python.exe
device: cuda
gpu_name: NVIDIA GeForce RTX 2050


,path_key,path_value
0,project_root,C:\Users\USER\Desktop\chess_engine
1,run_dir,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...
2,data_root,C:\Users\USER\Desktop\chess_engine\data\process
3,experiment_dir,C:\Users\USER\Desktop\chess_engine\experiments...
4,output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
5,plots_dir,C:\Users\USER\Desktop\chess_engine\experiments...
6,reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
7,checkpoints_dir,C:\Users\USER\Desktop\chess_engine\experiments...
8,cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...
9,runs_dir,C:\Users\USER\Desktop\chess_engine\experiments...


,is_valid,name,target_scale,sampler_mode,loss_mode
0,True,A1_curvature_compensated,600.0,random,curvature_compensated
1,True,A2_band_balanced,600.0,band_balanced,baseline_hybrid
2,True,C1_scale800,800.0,random,baseline_hybrid
3,True,C2_scale1200,1200.0,random,baseline_hybrid


## Self-Check And Gradient Audit

Cell nay fail-fast cho cac loi so dang:

- checkpoint model khong con `forward_logits`
- head khong dung `tanh`
- remap target `600 -> c` sai tinh chat monotonic / identity
- `oracle_subset_rows.csv` khong map dung ve shard goc
- balanced sampler duplicate hoac bo mat sample

Sau do no xuat `gradient_mass_audit` cho baseline va cac variant.

In [2]:
benchmark = lab.benchmark_single_train_step(
    init_ckpt_path=CHECKPOINT,
    data_root=paths["data_root"],
    device=DEVICE,
    batch_size=TRAIN_CFG.batch_size,
    num_shards=TRAIN_CFG.benchmark_num_shards,
)
checkpoint_validation = lab.validate_checkpoint_model(CHECKPOINT, device=DEVICE)
remap_validation = lab.validate_target_remap_logic([600.0, 800.0, 1200.0])
oracle_mapping_validation = lab.validate_oracle_subset_mapping(ORACLE_CFG, data_root=paths["data_root"])
sampler_validation = lab.validate_band_balanced_sampler(batch_size=TRAIN_CFG.batch_size)
split_rows = [
    lab.summarize_split_layout(paths["data_root"], split_name)
    for split_name in ("train", "val", "test")
]

lab.save_json(benchmark, paths["reports_dir"] / "runtime_benchmark.json")
lab.save_json(checkpoint_validation, paths["reports_dir"] / "checkpoint_validation.json")
lab.save_json(remap_validation, paths["reports_dir"] / "target_remap_validation.json")
lab.save_json(oracle_mapping_validation, paths["reports_dir"] / "oracle_subset_mapping_validation.json")
lab.save_json(sampler_validation, paths["reports_dir"] / "band_sampler_validation.json")
lab.save_dataframe(pd.DataFrame(split_rows), paths["reports_dir"] / "split_summary.csv")

gradient_tables = []
for variant_name in ["A1_curvature_compensated", "A2_band_balanced", "C1_scale800", "C2_scale1200", "B1_center_penalty"]:
    variant = VARIANT_DICT[variant_name]
    grad_df = lab.compute_gradient_mass_profile(
        data_root=paths["data_root"],
        split="train",
        variant=variant,
        num_shards=None,
    )
    grad_df.insert(0, "variant", variant.name)
    gradient_tables.append(grad_df)
    lab.save_dataframe(grad_df, paths["reports_dir"] / f"gradient_mass_{variant.name}.csv")
gradient_summary = pd.concat(gradient_tables, axis=0, ignore_index=True)
lab.save_dataframe(gradient_summary, paths["reports_dir"] / "gradient_mass_summary.csv")

display(pd.DataFrame([benchmark]))
display(pd.DataFrame([checkpoint_validation]))
display(pd.DataFrame([remap_validation]))
display(pd.DataFrame([oracle_mapping_validation]))
display(pd.DataFrame([sampler_validation]))
display(pd.DataFrame(split_rows))
display(gradient_summary)

,batch_size,step_time_sec,peak_mem_gb,steps_per_epoch_estimate,epoch_hours_estimate
0,640,1.854002,3.60046,6250,3.218754


,checkpoint,model_class,head_output_mode,has_forward_logits,checkpoint_epoch
0,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,DGRNChessNetV2,tanh,True,1


,probe,scales,monotonic,antisymmetric,identity_at_600
0,"[-0.95, -0.7, -0.2, -0.05, 0.0, 0.05, 0.2, 0.7...","[600.0, 800.0, 1200.0]","{'600.0': True, '800.0': True, '1200.0': True}","{'600.0': True, '800.0': True, '1200.0': True}",True


,checked_rows,mismatches,mismatch_rows
0,24,0,[]


,n,unique_ok,first_indices
0,67,True,"[10, 13, 56, 35, 2, 25, 7, 37, 46, 63, 38, 26,..."


,split,num_shards,samples
0,train,80,4000000
1,val,10,500000
2,test,10,500000


,variant,band_idx,band_label_y600,sample_count,sample_share,mean_abs_y,mean_y_curvature,mean_z_weight,mean_effective_y_factor,effective_gradient_mass_share
0,A1_curvature_compensated,0,"[0.000,0.050]",1257236,0.314309,0.015722,0.998990,1.000000,1.076498,0.329938
1,A1_curvature_compensated,1,"[0.050,0.200]",942764,0.235691,0.109487,0.972795,1.000000,1.076454,0.247404
2,A1_curvature_compensated,2,"[0.200,0.500]",728000,0.182000,0.340282,0.771934,1.000000,1.076422,0.191041
3,A1_curvature_compensated,3,"[0.500,0.700]",512000,0.128000,0.605916,0.401157,1.000000,1.076281,0.134347
4,A1_curvature_compensated,4,"[0.700,1.000]",560000,0.140000,0.821425,0.123370,1.000000,0.530455,0.097271
5,A2_band_balanced,0,"[0.000,0.050]",1257236,0.314309,0.015722,0.998990,0.999494,0.998990,0.416895
6,A2_band_balanced,1,"[0.050,0.200]",942764,0.235691,0.109487,0.972795,0.986252,0.972795,0.304461
7,A2_band_balanced,2,"[0.200,0.500]",728000,0.182000,0.340282,0.771934,0.876452,0.771934,0.186787
8,A2_band_balanced,3,"[0.500,0.700]",512000,0.128000,0.605916,0.401157,0.629503,0.401157,0.068564
9,A2_band_balanced,4,"[0.700,1.000]",560000,0.140000,0.821425,0.123370,0.317568,0.123370,0.023293


## Baseline Snapshot

Baseline duoc evaluate lai bang:

- scale-aware val/test metrics trong mien chuan `y600`
- oracle-subset metrics tren cung subset diagnostic, cung quy ve `y600` de so sanh cong bang voi cac run scale khac

Tat ca run ablation sau do se duoc so voi baseline nay.

In [3]:
ORACLE_BUNDLE = lab.load_oracle_subset_bundle(ORACLE_CFG, data_root=paths["data_root"])

baseline = lab.baseline_snapshot(
    init_ckpt_path=CHECKPOINT,
    data_root=paths["data_root"],
    oracle_bundle=ORACLE_BUNDLE,
    oracle_cfg=ORACLE_CFG,
    train_cfg=TRAIN_CFG,
    paths=paths,
    device=DEVICE,
    target_scale=600.0,
)

baseline_preview = pd.DataFrame(
    [
        {
            "label": "baseline",
            "val_mse_0.7eq": baseline["val_eval"]["metrics"]["bands"]["0.70"]["mse"],
            "test_mse_0.7eq": baseline["test_eval"]["metrics"]["bands"]["0.70"]["mse"],
            "oracle_stable_0.7_slope": baseline["oracle_eval"]["summary"]["stable_0.7_slope"],
            "oracle_center_amp_ratio": baseline["oracle_eval"]["summary"]["center_amp_ratio_eq_0.05"],
            "oracle_center_false_0.1eq": baseline["oracle_eval"]["summary"]["center_false_pred0.1eq"],
            "oracle_center_false_0.2eq": baseline["oracle_eval"]["summary"]["center_false_pred0.2eq"],
            "oracle_midband_mae_sum_stable": baseline["oracle_eval"]["summary"]["midband_teacher_vs_oracle_mae_sum_stable"],
            "oracle_gate_score": lab.oracle_gate_score(baseline["oracle_eval"]["summary"]),
        }
    ]
)
display(baseline_preview)

[val_600] offset=0 / 100000 elapsed=3.3s
[val_600] offset=25600 / 100000 elapsed=77.6s
[val_600] offset=51200 / 100000 elapsed=154.2s
[val_600] offset=76800 / 100000 elapsed=231.8s
[test_600] offset=0 / 200000 elapsed=3.2s
[test_600] offset=25600 / 200000 elapsed=83.8s
[test_600] offset=51200 / 200000 elapsed=164.7s
[test_600] offset=76800 / 200000 elapsed=244.8s
[test_600] offset=102400 / 200000 elapsed=325.0s
[test_600] offset=128000 / 200000 elapsed=400.7s
[test_600] offset=153600 / 200000 elapsed=476.8s
[test_600] offset=179200 / 200000 elapsed=552.5s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s


,label,val_mse_0.7eq,test_mse_0.7eq,oracle_stable_0.7_slope,oracle_center_amp_ratio,oracle_center_false_0.1eq,oracle_center_false_0.2eq,oracle_midband_mae_sum_stable,oracle_gate_score
0,baseline,0.050984,0.051428,0.575122,5.851215,0.558824,0.205882,0.598789,1.042111


## Run Selected Variants

Cell nay chay fine-tune ngan cho cac variant da chon.

Luu y:
- `A1` nham kiem tra curvature-compensated loss
- `A2` nham kiem tra density skew / sampler
- `C1/C2` nham kiem tra scale mismatch
- `B1` da duoc implement trong helper, nhung khong chay mac dinh o notebook nay

In [4]:
variant_runs = []
for variant_name in SELECTED_VARIANTS:
    variant = VARIANT_DICT[variant_name]
    print(f"\n===== RUN {variant.name} =====")
    print(variant.description)
    result = lab.run_variant_finetune(
        init_ckpt_path=CHECKPOINT,
        data_root=paths["data_root"],
        variant=variant,
        train_cfg=TRAIN_CFG,
        oracle_cfg=ORACLE_CFG,
        oracle_bundle=ORACLE_BUNDLE,
        paths=paths,
        device=DEVICE,
    )
    variant_runs.append(result)


===== RUN A1_curvature_compensated =====
Curvature-compensated loss to test Failure A directly.
[A1_curvature_compensated] finished shard 1/80
[A1_curvature_compensated] finished shard 2/80
[A1_curvature_compensated][epoch=0] step=200/6250 obj=0.068773 y_term=0.068492 z_term=0.069295 center_pen=0.000000
[A1_curvature_compensated] finished shard 3/80
[A1_curvature_compensated] finished shard 4/80
[A1_curvature_compensated] finished shard 5/80
[A1_curvature_compensated][epoch=0] step=400/6250 obj=0.069046 y_term=0.068599 z_term=0.069877 center_pen=0.000000
[A1_curvature_compensated] finished shard 6/80
[A1_curvature_compensated] finished shard 7/80
[A1_curvature_compensated][epoch=0] step=600/6250 obj=0.068981 y_term=0.068481 z_term=0.069911 center_pen=0.000000
[A1_curvature_compensated] finished shard 8/80
[A1_curvature_compensated] finished shard 9/80
[A1_curvature_compensated] finished shard 10/80
[A1_curvature_compensated][epoch=0] step=800/6250 obj=0.068864 y_term=0.068375 z_term=0

## Compare Results

Bieu bang cuoi cung dat nhung metric gate quan trong can doi:

- tat ca cot compare deu o mien chuan `y600`, khong do truc tiep tren thang rieng cua tung variant
- `oracle_stable_0.7_slope`
- `oracle_midband_mae_sum_stable`
- `oracle_center_amp_ratio`
- `oracle_center_false_0.1eq`
- `oracle_center_false_0.2eq`
- `oracle_gate_score`

In [5]:
compare_df = lab.compare_runs_table(baseline, variant_runs)
lab.save_dataframe(compare_df, paths["reports_dir"] / "compare_runs.csv")
display(compare_df.sort_values("oracle_gate_score", ascending=True))

,label,target_scale,test_mse_0.7eq,test_slope_0.7eq,oracle_stable_0.7_slope,oracle_midband_mae_sum_stable,oracle_center_amp_ratio,oracle_center_false_0.1eq,oracle_center_false_0.2eq,oracle_gate_score,delta_gate_vs_baseline
1,A1_curvature_compensated,600.0,0.059683,0.682593,0.646691,0.565637,7.150632,0.588235,0.294118,1.009938,-0.032172
2,A2_band_balanced,600.0,0.050254,0.594372,0.569950,0.591547,5.551442,0.558824,0.205882,1.037454,-0.004656
4,C2_scale1200,1200.0,0.059608,0.639198,0.607677,0.590804,6.680769,0.588235,0.235294,1.039906,-0.002204
3,C1_scale800,800.0,0.054289,0.629565,0.601030,0.588064,6.152091,0.617647,0.176471,1.040490,-0.001621
0,baseline,600.0,0.051428,0.605962,0.575122,0.598789,5.851215,0.558824,0.205882,1.042111,0.000000
